# Sleep-EDF stage-label upgrade and feature preparation

The default path asks you to upload the existing schema-v1 feature archive, downloads only the expert hypnograms, and adds five-class stage labels. It preserves the existing EEG features, forecast targets, and subject-wise train/validation/test assignments. It checks every new stage label against the archived wake/sleep label.

To rebuild the EEG features instead, set `UPGRADE_EXISTING_ARCHIVE = False`. That path downloads the paired PSG recordings and hypnograms. `MAX_SUBJECTS` applies only to this full rebuild; the upgrade path always uses every recording present in your uploaded archive.

An awakening starts at a transition from scored sleep into **at least two consecutive minutes labeled Wake**. Each timing example uses only the **five minutes of EEG available at the prediction time**. Later expert annotations supply the forecast target and are not part of model input.

[Sleep-EDF Expanded on PhysioNet](https://physionet.org/content/sleep-edfx/1.0.0/). The full dataset contains 153 Sleep Cassette recordings from 78 people.


In [ ]:
%pip -q install "mne>=1.7,<2"

## 1. Configuration

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.request import urlopen
import hashlib
import re
import shutil

import mne
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from scipy.integrate import trapezoid
from scipy.signal import butter, sosfiltfilt, welch
from sklearn.model_selection import train_test_split

SEED = 42
MAX_SUBJECTS = None      # None = full Sleep Cassette. Set to 10 for a small run.
UPGRADE_EXISTING_ARCHIVE = True  # Add hypnogram stage labels without recomputing EEG features.
DOWNLOAD_WORKERS = 4
KEEP_RAW_FILES = False  # Keep newly downloaded EDFs after extracting features.
CHANNEL = "EEG Fpz-Cz"
FS = 100
EPOCH_SEC = 30
EPOCH_SAMPLES = FS * EPOCH_SEC
HISTORY_EPOCHS = 10
WAKE_EPOCHS = 4
HORIZON_MIN = 60
HORIZON_EPOCHS = HORIZON_MIN * 60 // EPOCH_SEC
FORECAST_STRIDE_EPOCHS = 4
AWAKE, ASLEEP, UNKNOWN = 0, 1, -1

WORK_DIR = Path("/content/sleep_edf_wakeup")
RAW_DIR = WORK_DIR / "raw"
FEATURE_DIR = WORK_DIR / "features"
for directory in (RAW_DIR, FEATURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

## 2. Download the EEG and expert hypnograms, then split by person

Each pair contains one PSG file and its matching annotation file. Both nights from the same person always go into the same split.

In [ ]:
PHYSIONET = "https://physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/"
MIRROR = "https://physionet-open.s3.amazonaws.com/sleep-edfx/1.0.0/sleep-cassette/"

def recording_manifest():
    with urlopen(PHYSIONET, timeout=40) as response:
        listing = response.read().decode()
    entries = re.findall(
        r'href="(SC4\d{2}[12][A-Z0-9]{2}-(?:PSG|Hypnogram)\.edf)".*?(\d+)\s*$',
        listing, flags=re.MULTILINE,
    )
    paired = {}
    for filename, size in entries:
        record = filename[:6]
        kind = "psg" if filename.endswith("-PSG.edf") else "hyp"
        paired.setdefault(record, {})[kind] = (filename, int(size))
    if len(paired) != 153 or any(set(pair) != {"psg", "hyp"} for pair in paired.values()):
        raise RuntimeError("The PhysioNet index did not contain the expected 153 complete pairs.")
    rows = [
        {"record": record, "subject": record[:5], **paths}
        for record, paths in sorted(paired.items())
    ]
    if len({row["subject"] for row in rows}) != 78:
        raise RuntimeError("Unexpected subject count in the Sleep Cassette index.")
    return rows

def download_file(entry):
    name, size = entry
    path = RAW_DIR / name
    if path.exists() and path.stat().st_size == size:
        return path
    temporary = path.with_suffix(".part")
    for base in (MIRROR, PHYSIONET, MIRROR):
        try:
            with urlopen(base + name, timeout=90) as response, temporary.open("wb") as stream:
                shutil.copyfileobj(response, stream)
            if temporary.stat().st_size != size:
                raise IOError(f"Incomplete download for {name}")
            temporary.replace(path)
            return path
        except (OSError, ValueError) as error:
            last_error = error
            temporary.unlink(missing_ok=True)
    raise RuntimeError(f"Could not download {name}: {last_error}") from last_error

def split_by_subject(rows, limit):
    subjects = sorted({row["subject"] for row in rows})
    if limit is not None:
        if not 5 <= limit <= len(subjects):
            raise ValueError(f"MAX_SUBJECTS must be between 5 and {len(subjects)}.")
        subjects = subjects[:limit]
    train, remaining = train_test_split(subjects, test_size=0.4, random_state=SEED)
    validation, test = train_test_split(remaining, test_size=0.5, random_state=SEED)
    subject_splits = {"train": train, "validation": validation, "test": test}
    split_for = {subject: label for label, group in subject_splits.items() for subject in group}
    selected = [
        {**row, "split": split_for[row["subject"]]}
        for row in rows if row["subject"] in split_for
    ]
    assert len(split_for) == len(subjects)
    return selected, subject_splits

all_records = recording_manifest()
if UPGRADE_EXISTING_ARCHIVE:
    records = all_records
    subject_splits = None
    print(f"Found {len(records)} recordings. The upgrade path will retain split assignments from the uploaded archive.")
else:
    records, subject_splits = split_by_subject(all_records, MAX_SUBJECTS)
    for split, subjects in subject_splits.items():
        print(f"{split:>10}  {len(subjects):2d} people, {sum(row['split'] == split for row in records):3d} nights")


## 3. Align the expert sleep labels and prepare EEG features

The EEG channel is sampled at 100 Hz. Each 30-second epoch contains 3,000 samples. N1, N2, stages 3 and 4, and REM all count as **Asleep (1)**. An expert Wake label becomes **Awake (0)**. Epochs without a complete scored label remain unknown.

For every EEG epoch, calculate 11 features covering absolute and relative power in five frequency bands and RMS amplitude. Filtering and feature extraction operate within the 30-second epoch. The saved timestamp and recording ID identify the original waveform if it is needed later.

In [ ]:
STAGE_CODES = {"Sleep stage W": 0, "Sleep stage 1": 1, "Sleep stage 2": 2,
               "Sleep stage 3": 3, "Sleep stage 4": 3, "Sleep stage R": 4}
BANDS_HZ = ((0.5, 4), (4, 8), (8, 12), (12, 16), (16, 30))
BAND_NAMES = ("delta", "theta", "alpha", "sigma", "beta")
FEATURE_NAMES = tuple(f"{band}_log_power" for band in BAND_NAMES) + tuple(
    f"{band}_relative_power" for band in BAND_NAMES
) + ("log_rms",)
FEATURE_COUNT = len(FEATURE_NAMES)
SOS = butter(4, [0.5, 30], btype="bandpass", fs=FS, output="sos")

def read_eeg(path):
    raw = mne.io.read_raw_edf(path, include=[CHANNEL], preload=False, verbose="ERROR")
    if raw.ch_names != [CHANNEL] or not np.isclose(raw.info["sfreq"], FS):
        raise ValueError(f"Expected {CHANNEL} at {FS} Hz in {path.name}.")
    return raw

def stage_labels(annotations, epoch_count, first_time=0):
    stages = np.full(epoch_count, -1, dtype=np.int8)
    for annotation in annotations:
        stage = STAGE_CODES.get(annotation["description"])
        if stage is None:
            continue
        start = annotation["onset"] - first_time
        stop = start + annotation["duration"]
        first = max(0, int(np.ceil((start - 1e-5) / EPOCH_SEC)))
        last = min(epoch_count, int(np.floor((stop + 1e-5) / EPOCH_SEC)))
        if first < last:
            if np.any(stages[first:last] != -1):
                raise ValueError("Overlapping full-epoch sleep-stage annotations.")
            stages[first:last] = stage
    return stages

def epoch_states(raw, annotation_path):
    raw.set_annotations(mne.read_annotations(annotation_path), emit_warning=False)
    stages = stage_labels(raw.annotations, raw.n_times // EPOCH_SAMPLES, raw.first_time)
    states = np.full(len(stages), UNKNOWN, dtype=np.int8)
    known = stages >= 0
    states[known] = np.where(stages[known] == 0, AWAKE, ASLEEP)
    return states, stages

def eeg_features(epochs_volts):
    epochs = np.asarray(epochs_volts, dtype=np.float64)
    if epochs.ndim != 2 or epochs.shape[1] != EPOCH_SAMPLES or not np.isfinite(epochs).all():
        raise ValueError(f"Provide finite EEG with shape (epochs, {EPOCH_SAMPLES}) in volts.")
    filtered = sosfiltfilt(SOS, epochs * 1e6, axis=-1)
    frequencies, power = welch(filtered, fs=FS, nperseg=2*FS, axis=-1)
    powers = []
    for low, high in BANDS_HZ:
        band = (frequencies >= low) & (frequencies <= high)
        powers.append(trapezoid(power[:, band], x=frequencies[band], axis=-1))
    absolute = np.column_stack(powers)
    relative = absolute / np.maximum(absolute.sum(axis=1, keepdims=True), 1e-12)
    rms = np.sqrt(np.mean(filtered**2, axis=1))
    return np.column_stack((np.log1p(absolute), relative, np.log1p(rms))).astype(np.float32)

feature_spec = repr((CHANNEL, FS, EPOCH_SEC, BANDS_HZ, "butter4-0.5-30-welch2s-rms-v1"))
feature_version = hashlib.sha256(feature_spec.encode()).hexdigest()[:10]

def prepare_record(row):
    cache = FEATURE_DIR / f"{row['record']}_{feature_version}.npz"
    if cache.exists():
        with np.load(cache) as data:
            features, states = data["features"], data["states"]
            if "stages" in data:
                return features, states, data["stages"]
        hyp = download_file(row["hyp"])
        stages = stage_labels(mne.read_annotations(hyp), len(states))
        if not np.array_equal(states != UNKNOWN, stages >= 0) or not np.array_equal(states[stages >= 0],
                np.where(stages[stages >= 0] == 0, AWAKE, ASLEEP)):
            raise ValueError(f"Sleep-stage annotations disagree with cached labels for {row['record']}")
        temporary = cache.with_suffix(".tmp.npz")
        np.savez_compressed(temporary, features=features, states=states, stages=stages)
        temporary.replace(cache)
        return features, states, stages

    source_entries = (row["psg"], row["hyp"])
    newly_downloaded = [
        RAW_DIR / name for name, size in source_entries
        if not (RAW_DIR / name).exists() or (RAW_DIR / name).stat().st_size != size
    ]
    psg, hyp = map(download_file, source_entries)
    raw = read_eeg(psg)
    states, stages = epoch_states(raw, hyp)
    features = np.empty((len(states), FEATURE_COUNT), dtype=np.float32)
    for start in range(0, len(states), 128):
        stop = min(start + 128, len(states))
        eeg = raw.get_data(start=start * EPOCH_SAMPLES, stop=stop * EPOCH_SAMPLES)[0]
        features[start:stop] = eeg_features(eeg.reshape(-1, EPOCH_SAMPLES))
    if not np.any(states == ASLEEP):
        raise ValueError(f"No scored sleep could be aligned for {row['record']}.")
    temporary = cache.with_suffix(".tmp.npz")
    np.savez_compressed(temporary, features=features, states=states, stages=stages)
    temporary.replace(cache)
    if not KEEP_RAW_FILES:
        for path in newly_downloaded:
            path.unlink(missing_ok=True)
    return features, states, stages

def upgrade_existing_archive(path, rows):
    """Add expert stages while preserving every array and the assigned subject splits."""
    with np.load(path, allow_pickle=False) as saved:
        original = {key: saved[key] for key in saved.files}
    if int(original["schema_version"]) != 1 or "y_stage" in original:
        raise ValueError("Choose the existing schema-v1 binary archive for a stage-label upgrade.")
    by_record = {row["record"]: row for row in rows}
    record_ids = original["record_id"]
    if set(record_ids) != set(by_record):
        raise ValueError("The uploaded archive and Sleep-EDF recording index do not contain the same recordings.")
    if not np.array_equal(original["record_subject_id"], np.asarray([by_record[r]["subject"] for r in record_ids])):
        raise ValueError("Recording-to-subject assignments disagree with the existing archive.")

    labels = np.empty(len(original["y_state"]), dtype=np.int8)
    for record in record_ids:
        rows_for_record = np.flatnonzero(original["state_record_id"] == record)
        ends = original["state_epoch_end_sec"][rows_for_record]
        annotations = mne.read_annotations(download_file(by_record[record]["hyp"]))
        stages = stage_labels(annotations, int(ends.max() // EPOCH_SEC))
        epoch_indices = ends // EPOCH_SEC - 1
        aligned = stages[epoch_indices]
        if np.any(aligned < 0):
            raise ValueError(f"Some scored epochs have no full-epoch stage label in {record}.")
        if not np.array_equal(original["y_state"][rows_for_record], (aligned != 0).astype(original["y_state"].dtype)):
            raise ValueError(f"Hypnogram labels disagree with the archive's wake/sleep labels for {record}.")
        labels[rows_for_record] = aligned
        print(f"Aligned five-class stages for {record}", flush=True)

    upgraded = {**original, "y_stage": labels, "schema_version": np.asarray(2, dtype=np.int64)}
    manifest = {"record_id": original["record_id"]}
    for key, value in original.items():
        if key.startswith("record_") and key != "record_id":
            manifest[key.removeprefix("record_")] = value
    return upgraded, pd.DataFrame(manifest)


## 4. Build time-to-awakening targets

The input window uses the latest **ten epochs, or five minutes**, ending on a scored sleep epoch. For example, if the window ends at 02:10 and a qualifying Wake interval starts at 02:22, the target is **12 minutes**. We take a candidate every two minutes.

If the annotations confirm there was no qualifying onset in the next hour, the target is capped at **60 minutes**. An onset at the end of the hour counts as capped. A window with insufficient future annotation to make either determination is excluded. The archive includes the actual onset time for events observed before the horizon, and uses `-1` for capped rows.

In [ ]:
def sustained_wake_onsets(states):
    onsets = []
    for start in range(1, len(states)-WAKE_EPOCHS+1):
        if states[start-1] == ASLEEP and np.all(states[start:start+WAKE_EPOCHS] == AWAKE):
            onsets.append(start)
    return np.asarray(onsets, dtype=int)

def build_forecasts(features, states):
    history = sliding_window_view(states, HISTORY_EPOCHS)
    endpoints = np.arange(HISTORY_EPOCHS-1, len(states))
    eligible = (
        (states[endpoints] == ASLEEP)
        & np.all(history != UNKNOWN, axis=1)
        & (endpoints % FORECAST_STRIDE_EPOCHS == 0)
    )
    ends = endpoints[eligible]
    unknown = np.r_[np.flatnonzero(states == UNKNOWN), len(states)]
    events = np.r_[sustained_wake_onsets(states), len(states)]
    next_event = events[np.searchsorted(events, ends+1)]
    next_unknown = unknown[np.searchsorted(unknown, ends+1)]
    distance = next_event - (ends+1)
    observed = (next_event < next_unknown) & (distance < HORIZON_EPOCHS)
    # The last possible onset before the horizon needs four scored Wake epochs.
    known_horizon = next_unknown >= ends + HORIZON_EPOCHS + WAKE_EPOCHS
    keep = observed | known_horizon
    ends = ends[keep]
    target_minutes = np.where(observed[keep], distance[keep] * EPOCH_SEC / 60, HORIZON_MIN)
    indices = ends[:, None] - np.arange(HISTORY_EPOCHS-1, -1, -1)
    windows = features[indices]
    observed_onsets = np.where(
        observed[keep], next_event[keep] * EPOCH_SEC, -1
    ).astype(np.int32)
    return windows, target_minutes.astype(np.float32), ends, observed_onsets

def night_mask(states):
    asleep = np.flatnonzero(states == ASLEEP)
    padding = 30 * 60 // EPOCH_SEC
    start = max(0, asleep[0]-padding)
    stop = min(len(states), asleep[-1]+1+padding)
    mask = np.zeros(len(states), dtype=bool)
    mask[start:stop] = True
    return mask

# Small checks with known answers before downloading the full dataset.
example_states = np.r_[
    np.full(12, ASLEEP), np.full(3, AWAKE), np.full(10, ASLEEP),
    np.full(4, AWAKE), UNKNOWN, np.full(10, ASLEEP), np.full(4, AWAKE),
]
assert sustained_wake_onsets(example_states).tolist() == [25, 40]
example = build_forecasts(np.zeros((len(example_states), FEATURE_COUNT)), example_states)
assert dict(zip(example[2], example[1]))[16] == 4
long_sleep = np.full(HISTORY_EPOCHS + HORIZON_EPOCHS + WAKE_EPOCHS + 5, ASLEEP)
_, targets, ends, onsets = build_forecasts(np.zeros((len(long_sleep), FEATURE_COUNT)), long_sleep)
assert len(ends) and np.all(targets == HORIZON_MIN) and np.all(onsets == -1)
assert dict(zip(example[2], example[3]))[16] == 25 * EPOCH_SEC

In [ ]:
def prepare_examples(records):
    parts = {key: [] for key in (
        "X_state", "y_state", "state_record_id", "state_subject_id",
        "state_epoch_end_sec", "state_split",
        "X_forecast", "y_forecast_min", "forecast_record_id", "forecast_subject_id",
        "forecast_epoch_end_sec", "forecast_wake_onset_sec", "forecast_split",
    )}
    manifest = []

    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as workers:
        for number, (row, result) in enumerate(zip(records, workers.map(prepare_record, records)), 1):
            features, states, stages = result
            state_epochs = np.flatnonzero(night_mask(states) & (states != UNKNOWN))
            windows, target, ends, onsets = build_forecasts(features, states)
            record_id, subject, split = row["record"], row["subject"], row["split"]
            arrays = {
                "X_state": features[state_epochs],
                "y_state": states[state_epochs],
                "y_stage": stages[state_epochs],
                "state_record_id": np.repeat(record_id, len(state_epochs)),
                "state_subject_id": np.repeat(subject, len(state_epochs)),
                "state_epoch_end_sec": ((state_epochs+1) * EPOCH_SEC).astype(np.int32),
                "state_split": np.repeat(split, len(state_epochs)),
                "X_forecast": windows,
                "y_forecast_min": target,
                "forecast_record_id": np.repeat(record_id, len(ends)),
                "forecast_subject_id": np.repeat(subject, len(ends)),
                "forecast_epoch_end_sec": ((ends+1) * EPOCH_SEC).astype(np.int32),
                "forecast_wake_onset_sec": onsets,
                "forecast_split": np.repeat(split, len(ends)),
            }
            for key, values in arrays.items():
                parts[key].append(values)
            manifest.append({
                "record_id": record_id, "subject_id": subject, "split": split,
                "psg_filename": row["psg"][0], "hypnogram_filename": row["hyp"][0],
                "scored_state_epochs": len(state_epochs),
                "sustained_wake_events": len(sustained_wake_onsets(states)),
                "forecast_windows": len(ends),
                "forecast_windows_with_event": int(np.sum(onsets >= 0)),
            })
            if number % 10 == 0 or number == len(records):
                print(f"Prepared {number}/{len(records)} recordings", flush=True)
    data = {key: np.concatenate(values, axis=0) for key, values in parts.items()}
    return data, pd.DataFrame(manifest)

if UPGRADE_EXISTING_ARCHIVE:
    from google.colab import files
    print("Upload the existing schema-v1 sleep_edf_wake_data.npz archive.")
    uploaded = files.upload()
    archive_path = Path(next(name for name in uploaded if name.endswith(".npz")))
    data, manifest = upgrade_existing_archive(archive_path, records)
else:
    data, manifest = prepare_examples(records)
summary = manifest.groupby("split").agg(
    people=("subject_id", "nunique"),
    nights=("record_id", "count"),
    state_epochs=("scored_state_epochs", "sum"),
    sustained_wake_events=("sustained_wake_events", "sum"),
    forecast_windows=("forecast_windows", "sum"),
    forecast_windows_with_event=("forecast_windows_with_event", "sum"),
).reindex(["train", "validation", "test"])
display(summary)

## 5. Validate and export the prepared dataset

| Array | What it contains |
|---|---|
| `X_state`, `y_state`, `y_stage` | One 30-second feature row per scored epoch, with binary sleep/wake and five-class stage labels. Wake=0, N1=1, N2=2, N3=3 (merged stages 3 and 4), REM=4. |
| `X_forecast`, `y_forecast_min` | Ten chronologically ordered feature rows per five-minute window, with the target in minutes. |
| `state_*`, `forecast_*` | Recording ID, person ID, split, and window end time in seconds since the recording started. `forecast_wake_onset_sec` also gives onset time for observed events before the 60-minute horizon. |
| `record_*` | One row per recording with its original EDF filenames and event counts. |

The export retains the subject-wise split and the extracted feature values. During model development, use the split fields to select the distinct subject groups and fit any feature scaler on training data.

In [ ]:
def validate_examples(data, manifest):
    assert manifest["record_id"].is_unique
    assert manifest.groupby("subject_id")["split"].nunique().eq(1).all()
    split_labels = ["train", "validation", "test"]
    assert set(manifest["split"]) == set(split_labels)

    assert data["X_state"].shape == (len(data["y_state"]), FEATURE_COUNT)
    assert data["y_stage"].shape == data["y_state"].shape
    assert np.isin(data["y_stage"], np.arange(5)).all()
    assert np.array_equal(data["y_state"], data["y_stage"] != 0)
    assert data["X_forecast"].shape == (len(data["y_forecast_min"]), HISTORY_EPOCHS, FEATURE_COUNT)
    for task, labels in (("state", "y_state"), ("forecast", "y_forecast_min")):
        x = data[f"X_{task}"]
        assert np.isfinite(x).all()
        for name in ("record_id", "subject_id", "epoch_end_sec", "split"):
            assert data[f"{task}_{name}"].shape == data[labels].shape
        assert set(data[f"{task}_split"]) == set(split_labels)
        for split in split_labels:
            split_rows = data[f"{task}_split"] == split
            assert set(data[f"{task}_subject_id"][split_rows]).issubset(subject_splits[split])

    y = data["y_forecast_min"]
    end = data["forecast_epoch_end_sec"]
    onset = data["forecast_wake_onset_sec"]
    assert onset.shape == y.shape
    observed = onset >= 0
    assert np.all((y >= 0) & (y <= HORIZON_MIN))
    assert np.all(end >= HISTORY_EPOCHS * EPOCH_SEC)
    np.testing.assert_allclose(y[observed], (onset[observed] - end[observed]) / 60)
    assert np.all(y[observed] < HORIZON_MIN)
    assert np.all(onset[~observed] == -1) and np.all(y[~observed] == HORIZON_MIN)
    for split in split_labels:
        assert set(data["y_state"][data["state_split"] == split]) == {AWAKE, ASLEEP}
        assert np.any(observed & (data["forecast_split"] == split))

validate_examples(data, manifest)

archive = {
    **data,
    "feature_names": np.asarray(FEATURE_NAMES),
    "sampling_hz": np.int64(FS),
    "epoch_sec": np.int64(EPOCH_SEC),
    "history_epochs": np.int64(HISTORY_EPOCHS),
    "wake_minimum_sec": np.int64(WAKE_EPOCHS * EPOCH_SEC),
    "horizon_min": np.int64(HORIZON_MIN),
    "channel": np.asarray(CHANNEL),
    "schema_version": np.int64(2),
}
for column in manifest:
    values = manifest[column]
    key = column if column == "record_id" else f"record_{column}"
    archive[key] = values.to_numpy(dtype=str if values.dtype.kind in "OU" else np.int64)

export_path = WORK_DIR / "sleep_edf_wake_data.npz"
temporary = export_path.with_suffix(".tmp.npz")
np.savez_compressed(temporary, **archive)
with np.load(temporary, allow_pickle=False) as saved:
    assert set(saved.files) == set(archive)
    for key, expected in archive.items():
        np.testing.assert_array_equal(saved[key], expected)
temporary.replace(export_path)

print(f"Saved {export_path} ({export_path.stat().st_size / 1024**2:.1f} MiB)")
print(f"Five-class sleep-stage labels: {np.bincount(data['y_stage'], minlength=5)}")
print(f"Wake-up timing features: {data['X_forecast'].shape}")

To keep the prepared data after disconnecting from Colab, download `sleep_edf_wake_data.npz` from Colab’s Files panel. Load it later with `np.load(path, allow_pickle=False)`.